---
# <div style="text-align: center"> Introduction </div>
---

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System** class and its sources: the **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) Running <span style="color:blue">**SCOPE**</span> - Part 1: File Structure
7) Running <span style="color:blue">**SCOPE**</span> - Part 2: Execution 
8) Running <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions

---
# <div style="text-align: center"> Tutorial 8: Running SCOPE, part 3</div>
---

As part of the previous tutorial, you started the submission of three SCOPE tasks (tasks 1-3) for 4 geometries (geom1-4) of 3 systems (water, benzene, formaldehyde). In the best case, these are 36 computations. 

As discussed in Tutorial_6, being 3 tasks, you'll need to execute the commands below **at least** 4 times (3+1). 

```bash
scope run -n scope_env_cluster.npy -s 2-Systems/water/water.npy -t task1.scope task2.scope task3.scope -e 
scope run -n scope_env_cluster.npy -s 2-Systems/benzene/benzene.npy -t task1.scope task2.scope task3.scope -e 
scope run -n scope_env_cluster.npy -s 2-Systems/formaldehyde/formaldehyde.npy -t task1.scope task2.scope task3.scope -e
```

(1) The first time you execute this command, SCOPE will submit the computation(s) associated with <span style="color:orange">task1</span> to the cluster. The System file will be updated and saved   
(2) If you execute once <span style="color:orange">task1</span>'s computations finish, SCOPE will register <span style="color:orange">task1</span>, and prepare and submit <span style="color:red">task2</span>. The System file will be updated  
(3) If you execute once <span style="color:red">task2</span>'s computations finish, SCOPE will skip <span style="color:orange">task1</span> (it is already registered), register <span style="color:red">task2</span>, and submit <span style="color:maroon">task3</span>.  
(4) If you execute once <span style="color:maroon">task3</span>'s computations finish, SCOPE will skip <span style="color:orange">task1</span> and <span style="color:red">task2</span>, and register <span style="color:maroon">task3</span>. The System file will be updated, and the workflow will be over.


## 1) Summary of Actions

When executing "scope_run_task":

1. **Navigates the Workflow**: The script locates the Branch and Workflow(s) indicated in the input file, goes through the Jobs and Computations, and takes the appropriate actions: 
2. **Submits**: If the job has not been submitted, it creates the input and submission file and does the submission.
3. **Or Waits**: If the job is running, it does nothing
4. **Registers**: If the job has finished, scope *registers* it. That is, it reads the output and decides whether it finished 'well'
    1. If not, it takes appropriate action. For instance:
        1. If it is a GEOMETRY OPTIMIZATION that didn't reach a zero-gradient solution, it creates and submits a new computation that continues the previous run
        2. If it is an SCF computation in Quantum Espresso that didn't converge, it will modify some parameters and resubmit. In Gaussian, convergence failure is rare.
        3. If the computation was aborted by SLURM or for some other reason, the logfile will be discarded, and the computation will be resubmitted.
    2. If the job finished well, the logfile is parsed, and the relevant data is stored in the final state (fstate) defined in the input file.
       The job will be considered done, and even if you resubmit the same command, the computations won't run again. 

## 2) Actions in Detail

- We will now go through the details: the command "scope_run_task" is just the CLI form to call the function "run_task", which can be found in scope/utils.  
- If you open this function with a text editor, you'll identify commented blocks mentioning the STEPs. We'll go through these steps
- To make it easier, we will load the water system (you can load benzene or formaldehyde if you prefer).
- **IMPORTANT**: Make sure you load the System's binary file, once it went through the whole computational workflow or, at least, once task2 has finished. 

### Loads Example Case

In [1]:
import os
import scope

In [2]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('../')+'/Data/7-Tutorial_7'
## Loads the System object from a binary file. You can load Benzene or Formaldehyde systems instead. 
sys = scope.load_binary(f"{data_folder}/2-Systems/water/water.npy")

In [3]:
## We will discuss the PBE optimization job stored in our example system file. First, we navigate the workflow as discussed in Tutorial 2 (System -> Branch -> Workflow -> Job)
found, this_branch   = sys.find_branch("tutorial")
found, this_workflow = this_branch.find_workflow("geom1")
found, this_job      = this_workflow.find_job("pbe_opt")
## We will replicate the steps undergone by "scope_run_task" when running this Job.
print(this_job)

---------------------------------------------------
   >>> >>> >>> JOB                                 
---------------------------------------------------
 Source Type           = specie
 Source Name           = geom1
 Branch Name           = tutorial
 Workflow Name         = geom1
---------------------------------------------------
 Job path              = /home/svela/SCOPE/Tutorial/3-Computations/water/tutorial/
 Job keyword           = pbe_opt
 Job hierarchy         = 2
 Job requisites        = ['pbe_scf']
 Job constrains        = ['self']
 Job setup             = regular
 Num Computations      = 1
----------------------------------------------------
 self.isregistered (Temp) = True
 self.isgood       (Temp) = True
 self.isfinished   (Temp) = True




### Step 0:

In [4]:
# In Step 0, the function checks the existence of the files, and loads them. So nothing worth discussing here :)

### Step 1:

In [5]:
## In Step 1, it initiates the 'ith' LOOP for every task specified by the user. Notice that in the argument -t when calling "scope_run_task" you can specify many job_files if you wish. 
## Scope will iterate through those in order. 

## In 1.1, it reads the job_files and complements it with defaults if necessary
## In 1.2, it complements the global environment with any user choice specified in the "environment" of the job files. For instance, the variable "requested_procs"
## In 1.3, if the environment does not correspond to a computation cluster, or is poorly configured, scope disables submission, and input/submission files are not generated
## In 1.4 and 1.5, it finds the requested Branch and the Workflow. 
#                  If they do not exist yet, they are created. This is handy, since you don't need to initiate the classes in advance 

### Step 2:

In [6]:
## In 2.1, it finds or creates the Job using the "job_data" section in the job_file.
this_job.job_data  ## This is the job_data with which the job was created

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.branch              | <class 'str'>       | tutorial  
self.workflow            | <class 'list'>      | ['geom1', 'geom2', 'geom3', 'geom4']
self.job_setup           | <class 'str'>       | regular   
self.suffix              | <class 'str'>       | opt       
self.keyword             | <class 'str'>       | pbe_opt   
self.istate              | <class 'str'>       | initial   
self.fstate              | <class 'str'>       | pbe_opt   
self.hierarchy           | <class 'int'>       | 2         
self.requisites          | <class 'list'>      | ['pbe_scf']
self.constrains          | <class 'list'>      | ['self']  
self.must_be_good        | <class 'bool'>      | True      

In [7]:
## In 2.2, if the Job already exists, it checks whether the "job_data" was updated by the user since the job was created

In [8]:
## In 2.3, it checks whether the "job" requisites and constrains are fulfilled. If not, the job is skipped
print(this_job.requisites, this_job.requisites_fulfilled)
print(this_job.constrains, this_job.constrains_fulfilled)

## In this case, both were fulfilled

['pbe_scf'] True
['self'] True


### Step 3:

In [9]:
## In Step 3, it finds or creates the computations, using the variable "job_setup" in the "job_data" section of the job_file:
print(this_job.job_data.job_setup)

## There are 4 options for of job_setup:
##  - "regular" or "reg":          The most common. A Job that expects one computation
##  - "displacement" or "disp":    This job expects an initial state (istate) with vibrational normal modes. It will take those VNM with negative frequencies and prepare a new initial geometry that follows the modes (eigenvectors) 
##  - "findiff":                   This job will prepare 3N geometries using the VNM of the initial state to obtain the Hessian through a "finite differences" analysis. Each geometry will be associated with a state, and with a single-point computation
##  - "rep_opt":                   This job will prepare one geometry optimization computation, but the job will require energy convergence along several successive computations. This is useful for variable-cell optimizations in Quantum Espresso

regular


In [10]:
## In Step 3.1, it checks whether the "qc_data" was updated by the user, and whether the files (input, output, submission) exist
comp = this_job.computations[0]  ## This is the first and only computation associated with this job
print(comp.qc_data)              ## This is its QC_DATA 

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.software            | <class 'str'>       | g16       
self.jobtype             | <class 'str'>       | opt       
self.functional          | <class 'str'>       | pbe       
self.basis               | <class 'str'>       | sto-3g    
self.is_grimme           | <class 'bool'>      | True      
self.loose_opt           | <class 'bool'>      | True      
self.tight_opt           | <class 'bool'>      | False     
self.grimme_type         | <class 'str'>       | d2        
self.fstate              | <class 'str'>       | pbe_opt   
self.istate              | <class 'str'>       | initial   



In [11]:
## It also checks whether "continuation" computations exist. This is a very important piece of information if the system file is ever deleted or corrupted. 
## The way this is done brings us to the Filename class that we discussed in Tutorial 2:
print(comp.filename)
## Below are the ITEMS that were used to construct the name of this file. The ITEMS that are employed depend on the job_type 

sys_name: water, Format: water
sou_name: geom1, Format: geom1
suffix: opt, Format: opt
run_number: 1, Format: r1



In [12]:
## Basically, the Filename class enables SCOPE to identify whether continuation computations exists.
## A "continuation" computation basically shares the same name of the original one (that is, the same items in comp.filename) except for the "run_number". This is the run_number of our computation
print(comp.run_number)
## It indicates that it is the first attempt. The run number is the "_r1" part of the filename below
print(comp.inp_name)
## In this case, the computation worked at the first attempt, so there was no need for an update, meaning that a file with "_r2" does not exists. Hence, run_task detects that: 
print(comp.has_update)  
## If otherwise, if a "continuation" computation exists in the calculations folder, SCOPE will not create it once again, but will wait to register it. 

1
water_geom1_opt_r1.com
False


In [13]:
## In Step 3.2, it checks whether it must submit the computation. It will do so if:
## (1) The output file does not exist. 
## (2) The computation does not have a continuation (i.e. comp.has_update == False) 
## (3) The user wants to submit computations, as specified in the job_file. Variable "want_submit" in &options section of the job file. 
##     The default is True, but reverts to False if SLURM or SGE queue management is not detected
## (4) The job is not running already. This is evaluated by the environment using the "subprocess" module with predefined commands.
##     This check can be skipped by the variable "ignore_submitted" in the &options section of the job file. Not recommended.

## Also, if the output file exists, but not the input file, the function will show a warning prompting the user to investigate.

### Step 4:

In [14]:
## In step 4, if both input and output files exist, the computation might be "REGISTERED". Registering includes Parsing the output, and many other things 

In [16]:
## Step 4.1:
## - If the computation is already registered, this step will be skipped. The output parsing takes time so this step is skipped whenever possible. 
##   This is what would happen in "Case 4" of the example in cell number 3 of this notebook (see above):

## - If not, the computation is registered (see .Register() function of the Computation class in classes_workflow). 
## - Below is a summary of what it does:

## NOTE:
##  From here below, paths might be wrong if you switched computer (local vs. cluster) 
##  If necessary, do the following: comp.set_paths("/path_to_/3-Computations/water/tutorial")

##   - Files are checked: 
print("Input Exists:", comp.input_exists)
print("Stored Path:", comp.inp_path)
##   - The Output-class object is created:
print(comp.create_output())
##   - General Properties are registered
print("Finished:", comp.isfinished)
print("Elapsed Time:", comp.elapsed_time)
print("Status:", comp.status)

Input Exists: True
Stored Path: /Users/sergivela/Documents/SCOPE/Tutorials_Run/3-Computations/water/tutorial/water_geom1_opt_r1.com
---------------------------------------------------
          SCOPE Gaussian16 OUTPUT CLASS            
---------------------------------------------------
 #Lines           = 1684
 Job Type         = opt
 Requisites       = ['scf', 'opt']
---------------------------------------------------

Finished: True
Elapsed Time: 4.4
Status: worked


In [18]:
## Still in Step 4.1:
##   - The Energy is registered and stored in the final state.
found, istate = comp.source.find_state(comp.qc_data.istate)
found, fstate = comp.source.find_state(comp.qc_data.fstate)
print(istate.results["energy"], "in state:", istate.name)
print(fstate.results["energy"], "in state:", fstate.name)
##   - Because this is a job that implies a geometry optimization, the updated geometry is registered and stored in the final state.
print("Original coordinates of the first atom:", istate.coord[0], "in state:", istate.name)  ## from the initial state
print("Final coordinates of the first atom:   ", fstate.coord[0], "in state:", fstate.name)  ## from the final state

energy: -75.22540564 au in state: initial
energy: -75.23889404 au in state: pbe_opt
Original coordinates of the first atom: [0.0, 0.0, 0.0] in state: initial
Final coordinates of the first atom:    [-0.043628, -0.056241, -0.0] in state: pbe_opt


In [19]:
## In Step 4.2 it sets up continuation computations if needed.
## In Step 4.3 it handles repetitive jobs (only if this_job.job_setup = "rep_opt")
this_job.job_setup

'regular'

In [20]:
## In Step 4.4 it handles some errors for which a simple and routine solution exists. 
## This is done only if the flag "-e" is called when executing scope_run_job.

### End

- At the end of the *i-th* loop, if the system has been updated, it is saved (overwritten) where it belongs (the path is taken from *sys.system_file*).